In [ ]:
pip install -q paddleocr paddlepaddle==3.2.2 evaluate jiwer pandas pillow tqdm

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import evaluate
from jiwer import wer
from paddleocr import PaddleOCR

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_FOLDER = "/content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset"
TRAIN_FOLDER = BASE_FOLDER + "/Training"
VAL_FOLDER = BASE_FOLDER + "/Validation"
TEST_FOLDER = BASE_FOLDER + "/Testing"

In [ ]:
df_train = pd.read_csv(TRAIN_FOLDER + "/training_labels.csv")
df_train.drop(columns=["GENERIC_NAME"], inplace=True)
df_train

,IMAGE,MEDICINE_NAME
0,0.png,Aceta
1,1.png,Aceta
2,2.png,Aceta
3,3.png,Aceta
4,4.png,Aceta
...,...,...
3115,3115.png,Zithrin
3116,3116.png,Zithrin
3117,3117.png,Zithrin
3118,3118.png,Zithrin


In [ ]:
df_val = pd.read_csv(VAL_FOLDER + "/validation_labels.csv")
df_val.drop(columns=["GENERIC_NAME"], inplace=True)
df_val

,IMAGE,MEDICINE_NAME
0,0.png,Aceta
1,1.png,Aceta
2,2.png,Aceta
3,3.png,Aceta
4,4.png,Aceta
...,...,...
775,775.png,Zithrin
776,776.png,Zithrin
777,777.png,Zithrin
778,778.png,Zithrin


In [ ]:
df_test = pd.read_csv(TEST_FOLDER + "/testing_labels.csv")
df_test.drop(columns=["GENERIC_NAME"], inplace=True)
df_test

,IMAGE,MEDICINE_NAME
0,0.png,Aceta
1,1.png,Aceta
2,2.png,Aceta
3,3.png,Aceta
4,4.png,Aceta
...,...,...
775,775.png,Zithrin
776,776.png,Zithrin
777,777.png,Zithrin
778,778.png,Zithrin


In [ ]:
import os
import re
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import evaluate
from jiwer import wer
from paddleocr import PaddleOCR

BATCH_SIZE = 8

ocr = PaddleOCR(
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name="arabic_PP-OCRv5_mobile_rec",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_mobile_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_mobile_det`.
Creating model: ('arabic_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/arabic_PP-OCRv5_mobile_rec`.


In [ ]:
def normalize_text(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    return s

cer_metric = evaluate.load("cer")


def extract_text_from_paddle_result(result) -> str:
    texts = []

    def add_text(value):
        if value is None:
            return
        value = str(value).strip()
        if value:
            texts.append(value)

    def walk(obj):
        if obj is None:
            return

        if hasattr(obj, "rec_texts"):
            for t in getattr(obj, "rec_texts", []):
                add_text(t)
            return

        if isinstance(obj, dict):
            if "rec_texts" in obj:
                for t in obj["rec_texts"]:
                    add_text(t)
                return
            for v in obj.values():
                walk(v)
            return

        if isinstance(obj, (list, tuple)):
            for item in obj:
                if (
                    isinstance(item, (list, tuple))
                    and len(item) >= 2
                    and isinstance(item[1], (list, tuple))
                    and len(item[1]) >= 1
                    and isinstance(item[1][0], str)
                ):
                    add_text(item[1][0])
                else:
                    walk(item)
            return

        if isinstance(obj, str):
            add_text(obj)

    walk(result)
    return " ".join(texts).strip()


def paddle_predict_text(image: Image.Image) -> str:
    img_np = np.array(image)

    try:
        result = ocr.predict(img_np)
    except Exception:
        result = ocr.ocr(img_np, cls=False)

    return extract_text_from_paddle_result(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from PIL import Image
import os

TRAIN_IMG_DIR = os.path.join(TRAIN_FOLDER, "training_words")
VAL_IMG_DIR   = os.path.join(VAL_FOLDER,   "validation_words")
TEST_IMG_DIR  = os.path.join(TEST_FOLDER,  "testing_words")

SPLIT_TO_DIR = {
    "train": TRAIN_IMG_DIR,
    "val":   VAL_IMG_DIR,
    "test":  TEST_IMG_DIR,
}

def load_rgb(img_name: str, split_name: str):
    img_path = os.path.join(SPLIT_TO_DIR[split_name], img_name)
    return Image.open(img_path).convert("RGB")


def evaluate_split(df: pd.DataFrame, split_name: str):
    if len(df) == 0:
        return None, None

    preds_raw = []
    gts_raw = df["MEDICINE_NAME"].astype(str).tolist()

    for start in tqdm(range(0, len(df), BATCH_SIZE), desc=f"PaddleOCR {split_name}"):
        batch = df.iloc[start:start+BATCH_SIZE]
        images = [load_rgb(p, split_name) for p in batch["IMAGE"].tolist()]

        batch_preds = [paddle_predict_text(img) for img in images]
        preds_raw.extend(batch_preds)

    out = df.copy()
    out["split"] = split_name
    out["pred_raw"] = preds_raw
    out["pred_norm"] = out["pred_raw"].apply(normalize_text)
    out["gt_norm"] = out["MEDICINE_NAME"].apply(normalize_text)

    exact_match = (out["pred_norm"] == out["gt_norm"]).mean()
    cer_value = cer_metric.compute(
        predictions=out["pred_norm"].tolist(),
        references=out["gt_norm"].tolist()
    )
    wer_value = wer(out["gt_norm"].tolist(), out["pred_norm"].tolist())

    metrics = {
        "split": split_name,
        "samples": len(out),
        "exact_match": float(exact_match),
        "cer": float(cer_value),
        "wer": float(wer_value),
    }
    return metrics, out

all_metrics = []
all_outputs = []

for split_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    metrics, out = evaluate_split(df, split_name)
    if metrics is None:
        print(f"[WARN] No samples found for split: {split_name}")
        continue
    all_metrics.append(metrics)
    all_outputs.append(out)

metrics_df = pd.DataFrame(all_metrics)
print("\n===== SUMMARY =====")
print(metrics_df.to_string(index=False))


for out in all_outputs:
    split = out["split"].iloc[0]
    out_csv = os.path.join(BASE_FOLDER, f"paddleocr_predictions_{split}.csv")
    out.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

combined = pd.concat(all_outputs, ignore_index=True)
combined_csv = os.path.join(BASE_FOLDER, "paddleocr_predictions_all_splits.csv")
combined.to_csv(combined_csv, index=False)
print("Saved:", combined_csv)

metrics_csv = os.path.join(BASE_FOLDER, "paddleocr_metrics_summary.csv")
metrics_df.to_csv(metrics_csv, index=False)
print("Saved:", metrics_csv)

PaddleOCR test: 100%|██████████| 98/98 [07:47<00:00,  4.77s/it]



===== SUMMARY =====
split  samples  exact_match      cer      wer
train     3120     0.267628 0.319868 0.771203
  val      780     0.276923 0.348171 0.763291
 test      780     0.332051 0.263821 0.701266
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/paddleocr_predictions_train.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/paddleocr_predictions_val.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/paddleocr_predictions_test.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Prescription BD dataset/Doctor’s Handwritten Prescription BD dataset/paddleocr_predictions_all_splits.csv
Saved: /content/drive/MyDrive/Mixed OCR/English Medical/Doctor’s Handwritten Pres